# B7.2 — KLayout competitiveness audit

This notebook answers one decision question: **does the frozen B7.1 CNN provide a real end-to-end advantage over exact KLayout m1.2 checking?** KLayout remains the correctness oracle. The hard gate is 2× speedup at ≥99.5% recall, no registered severe/near-threshold miss, no clean-layout false alarm, and lower p95 latency.

The audit uses the same source and injected layouts. If the quality gate fails, the detector is archived and the project moves to B8.0 action-conditioned DRC prevention.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime, timezone
MY_DRIVE = Path('/content/drive/MyDrive')
PROJECT_DRIVE = MY_DRIVE / 'ADVLSI2 2026 Project'
assert MY_DRIVE.is_dir(), 'Google Drive did not mount at /content/drive/MyDrive.'
assert PROJECT_DRIVE.is_dir(), (
    f'Project root is missing: {PROJECT_DRIVE}. ' 
    'Do not create an empty replacement; move the canonical project folder into My Drive and rerun this cell.'
)
NOTEBOOKS_ROOT = PROJECT_DRIVE / 'notebooks'
EXPERIMENTS_ROOT = PROJECT_DRIVE / 'experiments'
B6_CHECKPOINTS = EXPERIMENTS_ROOT / 'B6_localization' / 'b6_multitask_unet'
B7_HISTORY = EXPERIMENTS_ROOT / 'B7_full_layout' / 'b7_full_layout'
CNN_ROOT = EXPERIMENTS_ROOT / 'B7_full_layout' / 'b7_2_cnn_gpu'
AUDIT_ROOT = EXPERIMENTS_ROOT / 'B7_full_layout' / 'b7_2_klayout_benchmark'
RUN_TAG = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
CNN_OUTPUT = CNN_ROOT / RUN_TAG
AUDIT_OUTPUT = AUDIT_ROOT / RUN_TAG
required_inputs = {
    'B6 checkpoints': B6_CHECKPOINTS,
    'B7 layout cache': B7_HISTORY / 'layout_cache',
}
missing_inputs = {name: path for name, path in required_inputs.items() if not path.is_dir()}
assert not missing_inputs, 'Missing canonical inputs: ' + ', '.join(
    f'{name}={path}' for name, path in missing_inputs.items()
)
for path in (CNN_OUTPUT, AUDIT_OUTPUT):
    path.mkdir(parents=True, exist_ok=True)
print('Project:', PROJECT_DRIVE)
print('B6 checkpoints:', B6_CHECKPOINTS)
print('B7 layout cache:', B7_HISTORY / 'layout_cache')
print('Run tag:', RUN_TAG)

In [ ]:
import os, shutil, subprocess, sys, tempfile
REPO = Path('/content/ADVLSI2_Project_updated')
REPO_URL = 'https://github.com/nocleo/ADVLSI2_Project_updated.git'
BRANCH = 'agent/research-impact-reset'
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO / 'requirements.txt')], check=True)
os.chdir(REPO)
print(subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

## 1. Fresh synchronized CNN timing on GPU

The historical B7.1 result was timed on CPU. This rerun uses CUDA, synchronizes GPU work at both timing boundaries, keeps the frozen model/policy, and reuses only the authoritative layout/report inputs—not old scan results.

In [ ]:
import json, torch
assert torch.cuda.is_available(), 'Select a Colab GPU runtime before running B7.2.'
shutil.copytree(B7_HISTORY / 'layout_cache', CNN_OUTPUT / 'layout_cache', dirs_exist_ok=True)
command = [
    sys.executable, 'scripts/run_b7_full_layout.py',
    '--checkpoint-dir', str(B6_CHECKPOINTS),
    '--output-dir', str(CNN_OUTPUT),
    '--device', 'cuda', '--batch-size', '64', '--phase', 'B7.1',
    '--segmentation-thresholds', '0.4',
    '--selection-objective', 'precision_at_recall',
    '--selection-minimum-recall', '0.95',
]
subprocess.run(command, check=True)
cnn_summary = json.loads((CNN_OUTPUT / 'summary.json').read_text())
{
    'device': cnn_summary['runtime']['device'],
    'validation_recall': cnn_summary['validation']['violation_recall'],
    'development_recall': cnn_summary['development_confirmation']['violation_recall'],
    'development_seconds': cnn_summary['development_confirmation']['end_to_end_seconds'],
}

## 2. Exact KLayout baseline

For each identical layout variant, KLayout is warmed once and measured five times. The batch total includes parse, recursive M1 materialization, exact 140 nm spacing check, and RDB writing. The script also measures rule-only timing after loading and 10 µm incremental regions.

In [ ]:
command = [
    sys.executable, 'scripts/run_b7_2_klayout_benchmark.py',
    '--b7-output-dir', str(CNN_OUTPUT),
    '--cnn-summary', str(CNN_OUTPUT / 'summary.json'),
    '--output-dir', str(AUDIT_OUTPUT),
    '--repeats', '5', '--warmups', '1',
    '--incremental-samples', '20',
]
subprocess.run(command, check=True)
audit = json.loads((AUDIT_OUTPUT / 'summary.json').read_text())
audit['comparison']

In [ ]:
import pandas as pd
rows = []
for group, result in audit['comparison'].items():
    rows.append({
        'group': group,
        'KLayout batch median (s)': result['klayout_batch_median_seconds'],
        'CNN end-to-end (s)': result['cnn_recorded_end_to_end_seconds'],
        'CNN/KLayout speedup': result['cnn_speedup_over_klayout'],
        'CNN recall': result['cnn_violation_recall'],
        'quality gate': result['quality_gate_passed'],
        'hard gate': result['hard_gate_passed'],
    })
display(pd.DataFrame(rows))
print('DECISION:', 'continue detector' if audit['hard_gate_passed'] else 'archive detector and start B8.0')

## 3. Show one 200×200 CNN input, exact mask, probability, and output

The model input is a 200×200 M1 raster. Its supervised output is the central 160×160 area; the surrounding 20-pixel halo supplies context without double-owning violations. The exact mask comes from KLayout, and the final candidate coordinates come from a local exact KLayout recovery query.

In [ ]:
import gzip, pickle, matplotlib.pyplot as plt, numpy as np
example_dir = CNN_OUTPUT / 'layouts' / 'development_confirmation' / 'injected__tt_um_2048_vga_game'
cache_path = CNN_OUTPUT / 'scan_cache' / 'injected__tt_um_2048_vga_game.pkl.gz'
with gzip.open(cache_path, 'rb') as handle:
    scan = pickle.load(handle)['scan']
sample = next(item for item in scan['diagnostics'] if item['owner_violation_ids'])
image, target, probability = sample['image'], sample['target'], sample['probability']
prediction = probability >= cnn_summary['deployment_policy']['segmentation_threshold']
figure, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(image, cmap='gray', origin='upper'); axes[0].set_title(f'CNN input {image.shape}')
axes[1].imshow(target, cmap='gray', origin='upper'); axes[1].set_title(f'Exact mask {target.shape}')
axes[2].imshow(probability, cmap='magma', origin='upper', vmin=0, vmax=1); axes[2].set_title('CNN probability')
overlay = np.zeros((*target.shape, 3), dtype=np.uint8)
overlay[target.astype(bool)] = [0, 220, 80]
overlay[prediction] = [255, 80, 80]
overlay[np.logical_and(target, prediction)] = [255, 220, 0]
axes[3].imshow(overlay, origin='upper'); axes[3].set_title('GT / thresholded output')
for axis in axes: axis.axis('off')
plt.tight_layout(); plt.show()
with (example_dir / 'exact_candidates.jsonl').open() as handle:
    example_candidate = json.loads(next(line for line in handle if line.strip()))
example_candidate

In [ ]:
archive = Path('/content/ADVLSI2_B7_2_results.zip')
with tempfile.TemporaryDirectory() as temp_dir:
    export = Path(temp_dir) / 'B7_2'
    export.mkdir()
    for name in ('summary.json', 'per_layout.jsonl', 'README.md'):
        shutil.copy2(AUDIT_OUTPUT / name, export / name)
    shutil.make_archive(str(archive.with_suffix('')), 'zip', export)
(CNN_ROOT / 'LATEST_RUN.txt').write_text(RUN_TAG + '\n')
(AUDIT_ROOT / 'LATEST_RUN.txt').write_text(RUN_TAG + '\n')
print(archive)